# **Proyecto de marketing bancario**
## Objetivo: de predecir si un cliente contratará un servicio

## Importamos todas las librerías que vamos a necesitar:

In [3]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

## Cargar los datos:
- Indicamos que los datos vienen separados por ; (no por , como suele ser habitual)

In [4]:
df = pd.read_csv("bank-full.csv", sep=";")

## Visualizacion en consola de una muestra de los datos:

In [5]:
#print("dataframe_head:\n", df.head())
#print("\n")
df.head()


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## Mostramos los tipos de datos de las features del df y las dimensiones (filas x columnas)

In [6]:
print("numero de filas: ", df.shape[0]) # tb vale len(df)
print("numero de columnas: ", df.shape[1])
print("\n")

print("df_types:\n", df.dtypes)
print("\n")

numero de filas:  45211
numero de columnas:  17


df_types:
 age           int64
job          object
marital      object
education    object
default      object
balance       int64
housing      object
loan         object
contact      object
day           int64
month        object
duration      int64
campaign      int64
pdays         int64
previous      int64
poutcome     object
y            object
dtype: object




## Mostramos las columnas (features) incluída la columna objetivo 'y':

In [7]:
print("df_columns:\n ", df.columns)
print("\n")

df_columns:
  Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='object')




## Mostramos info del dataframe:

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB


## Valores generales de dataframe (describe):

In [9]:
df.describe()

,age,balance,day,duration,campaign,pdays,previous
count,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000,45211.000000
mean,40.936210,1362.272058,15.806419,258.163080,2.763841,40.197828,0.580323
std,10.618762,3044.765829,8.322476,257.527812,3.098021,100.128746,2.303441
min,18.000000,-8019.000000,1.000000,0.000000,1.000000,-1.000000,0.000000
25%,33.000000,72.000000,8.000000,103.000000,1.000000,-1.000000,0.000000
50%,39.000000,448.000000,16.000000,180.000000,2.000000,-1.000000,0.000000
75%,48.000000,1428.000000,21.000000,319.000000,3.000000,-1.000000,0.000000
max,95.000000,102127.000000,31.000000,4918.000000,63.000000,871.000000,275.000000


## Checkeamos si hay datos corruptos (vacíos / NaN)...

In [10]:
print("Algun valor NaN o vacio? ", df.isnull().values.any())
print("Suma de valores NaN o vacio? ", df.isnull().values.sum())
print("\n")

Algun valor NaN o vacio?  False
Suma de valores NaN o vacio?  0




## Asignamos algunas constates:

In [11]:
RANDOM_STATE = 22 # Fijada semilla aleatoria

## Separación de variables:
- **X** Features
- **y** Target

In [12]:
X = df.drop("y", axis=1)
y = df["y"]
y

,y
0,no
1,no
2,no
3,no
4,no
...,...
45206,yes
45207,yes
45208,yes
45209,no


## Separar columnas:
- **numeric_features** columnas de tipo numérico
- **categorical_features** columnas de tipo categórico

In [13]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object", "string"]).columns

print("\nNumeric_features:\n", numeric_features.to_list())
print("\nCat_features:\n", categorical_features.to_list())


Numeric_features:
 ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']

Cat_features:
 ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


## Separación de los datos en: **Train / Test**

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

## Preprocesamiento **(numéricas)**

In [15]:
numeric_transformer = Pipeline([
    ("scaler", MinMaxScaler())
])

## Preprocesamiento **(categóricas)**

In [16]:
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("one_hot_encoder", OneHotEncoder(handle_unknown="ignore"))
])

## Transformación **(numéricas y categóricas)**

In [17]:
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

## Creamos el model **Decision Tree**, asignándo estos hiperparámetros:
- **param 1**: Aplicamos el parámetro clave: class_weight="balanced" para manejar el desbalanceo
- **param 2**: random_state=22 (para que el aleatorio sea siempre igual en cada ejecucion)

In [18]:
dt_model = Pipeline([
    ("prep", preprocessor),
    ("classifier", DecisionTreeClassifier(class_weight="balanced", random_state=22))
])

## Creamos el model **Random Forest**, asignando estos hiperparámetros:
- **param 1**: n_estimators entre 100-200 (suficientes para un buen resultado)
- **param 2** :  entre 0.2-0.5
- **param 3**: Aplicamos balanced para manejar el desbalanceo
- **param 4**: Random_state 22 para que NO cambie el aleatorio en cada ejecucion

In [19]:
rf_model = Pipeline([
    ("prep", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=150, min_samples_split=0.3, class_weight="balanced", random_state=22))
])

## **Creamos una función** para entrenar los diferentes modelos:
entrenar_modelo()

In [20]:
def entrenar_modelo(name, model):
  """
  Función para entrenar los diferentes modelos:
  - Recibe como parametros: una descripción del modelo y el propio modelo
  - Devuelve: una variable con las predicciones 'y_pred'
  """
  # Entrenar modelo:
  model.fit(X_train, y_train)

  # Predicciones:
  y_pred = model.predict(X_test)

  # Metricas (clasification_report):
  print("=" * 55)
  print(f"--- MÉTRICAS {name} ---")
  print("=" * 55)
  print(classification_report(y_test, y_pred))

  # Metricas (confusion-matrix):
  print(f"--- CONFUSION MATRIX ({name}) ---")
  print(confusion_matrix(y_test, y_pred))
  print("\n")

  return y_pred



## Invocamos la función creada anteriormente:
(Una vez por cada modelo que queramos entrenar)

In [21]:
entrenar_modelo("DECISION TREE", dt_model)
entrenar_modelo("RANDOM FOREST", rf_model)

--- MÉTRICAS DECISION TREE ---
              precision    recall  f1-score   support

          no       0.93      0.93      0.93      7971
         yes       0.47      0.47      0.47      1072

    accuracy                           0.88      9043
   macro avg       0.70      0.70      0.70      9043
weighted avg       0.88      0.88      0.88      9043

--- CONFUSION MATRIX (DECISION TREE) ---
[[7411  560]
 [ 567  505]]


--- MÉTRICAS RANDOM FOREST ---
              precision    recall  f1-score   support

          no       0.97      0.76      0.85      7971
         yes       0.31      0.80      0.45      1072

    accuracy                           0.77      9043
   macro avg       0.64      0.78      0.65      9043
weighted avg       0.89      0.77      0.81      9043

--- CONFUSION MATRIX (RANDOM FOREST) ---
[[6097 1874]
 [ 216  856]]




array(['no', 'no', 'no', ..., 'yes', 'yes', 'no'], dtype=object)

# **Conclusión Decision-Tree:**

Aún con el parametro class_weight=balanced al modelo le cuesta distinguir patrones de contratación. (con un f1-score de un 46%)

Sólo tiene **buen resultado** en patrones de **no contratación** al tener muchos mas ejemplos para aprender de ellos. (Aquí el f1-score llega hasta el 93%)

En cuanto a la matriz de confusión sólo consiguió encontrar a 491 dejando "escapar" a 581

# **Conclusión Random-Forest:**

Aunque el Decision Tree tiene mejor Accuracy y F1-Score, para una campaña de marketing el **modelo más útil es el Random Forest**, ya que su Recall del 82% nos asegura que el banco captará a la gran mayoría de clientes interesados, aunque a costa de hacer más llamadas a clientes que finalmente no contratarán.

En este modelo la matriz de confusión muestra que logró captar a 881 dejando "escapar" sólo a 191, mejorando muchísimo al otro modelo. Parece que este modelo es mejor para este ejercicio.